In [1]:
# imports
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit
from datetime import date, timedelta
import yfinance as yf
import shap
import matplotlib.pyplot as plt
import streamlit as st
import yfinance as yf
import plotly.graph_objects as go

### Training

In [2]:
def make_splits(df, n_splits=5, target_col='target', gap_days=21):
    """
    Create walk-forward splits from the merged DataFrame.
    gap_days: trading days between train end and test start (avoids lookahead).
    """
    # Work on unique dates to split time, not rows
    dates = df.index.get_level_values('Date').unique().sort_values()
    tss = TimeSeriesSplit(n_splits=n_splits, gap=gap_days)
    
    splits = []
    for train_idx, test_idx in tss.split(dates):
        train_dates = dates[train_idx]
        test_dates  = dates[test_idx]
        
        train_df = df[df.index.get_level_values('Date').isin(train_dates)]
        test_df  = df[df.index.get_level_values('Date').isin(test_dates)]
        
        splits.append((train_df, test_df))
    
    return splits

In [3]:
ai_sector_features_signals_df = pd.read_csv('../../data/ai_sector_features_df.csv', index_col=['Date', 'Ticker'], parse_dates=['Date'])

splits = make_splits(ai_sector_features_signals_df)
splits

[(                     ret_1d    ret_3d    ret_5d   ret_10d   ret_20d    vol_5d  \
  Date       Ticker                                                               
  2019-01-02 NVDA    0.020374  0.038499  0.071923 -0.051261 -0.166493  0.023554   
             TSM    -0.010567 -0.007069  0.034854  0.008283 -0.028465  0.020815   
             AVGO   -0.003028  0.010604  0.084303  0.008072  0.079068  0.025708   
             MRVL    0.019147  0.050955  0.145038  0.086345  0.028352  0.028901   
             INTC    0.003196  0.015531  0.080064  0.000000 -0.045224  0.024625   
  ...                     ...       ...       ...       ...       ...       ...   
  2020-02-03 NVDA    0.016495 -0.021219  0.000541 -0.035903  0.018046  0.026791   
             META    0.011292 -0.085293 -0.049704 -0.080805 -0.021469  0.037305   
             SMCI    0.025036 -0.004170 -0.001742 -0.000349  0.222696  0.018681   
             AMD     0.021702  0.010735 -0.025173 -0.057137 -0.011934  0.040794   
    

In [ ]:
# take fold's predictions and rescores them at a much finer resolution
# one IC number per trading day
def daily_ic_series(model, X_te, y_te, test_df):
    probs = model.predict_proba(X_te)[:, 1] # every row for predicting prob = 1 (outperform)
    temp = test_df.copy()
    temp['prob'] = probs
    temp['y'] = y_te.values

    daily_ic = temp.groupby(level='Date').apply(
        lambda g: np.corrcoef(g['prob'], g['y'])[0, 1] if g['y'].nunique() > 1 else np.nan
    )
    return daily_ic.dropna()

In [39]:
def train_evaluate(splits, feat_cols):
    all_results = []
    all_daily_ic = []
    for fold_num, (train_df, test_df) in enumerate(splits):
        # feat_cols = [
        #     'vol_5d', 'vol_20d', 'rsi_14', #'news_sentiment', 'pct_positive', 'news_count',
        #     'gpu_mentions', 'capex_up_score', #'capex_down_score',
        #     'capex_net', 'competitor_mentions', 'ai_sentence_ratio'
        # ]
        # use last 15% of training data for validation set
        train_dates = train_df.index.get_level_values('Date').unique().sort_values()
        cutoff = train_dates[int(len(train_dates) * 0.85)]
        fit_df = train_df[train_df.index.get_level_values('Date') <= cutoff]
        val_df = train_df[train_df.index.get_level_values('Date') > cutoff]
        
        X_fit, y_fit = fit_df[feat_cols], fit_df['Target']
        X_val, y_val = val_df[feat_cols], val_df['Target']
        X_te, y_te   = test_df[feat_cols], test_df['Target']

        asset_returns = test_df['ret_5d'].values

        model = lgb.LGBMClassifier(
            n_estimators=300, 
            learning_rate=0.05,
            num_leaves=31,    
            min_child_samples=20,
            subsample=0.8,    
            colsample_bytree=0.8,
            class_weight='balanced',
            reg_alpha=0.1,
            reg_lambda=0.1
        )
        model.fit(X_fit, y_fit,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        probs = model.predict_proba(X_te)[:,1]
        ic  = np.corrcoef(probs, y_te)[0,1]
        acc = accuracy_score(y_te, probs > 0.5)

        # volatile threshold
        vol_5d_thresh   = np.percentile(test_df['vol_5d'],   70)
        # vol_20d_thresh  = np.percentile(test_df['vol_20d'],  70)
        # vol_spike_thresh = np.percentile(test_df['vol_spike'], 70)

        # low_vol_mask = (
        #     (test_df['vol_5d'].values   < vol_5d_thresh) &
        #     (test_df['vol_20d'].values  < vol_20d_thresh) &
        #     (test_df['vol_spike'].values < vol_spike_thresh)
        # )

        # sharpe ratio
        # signals = np.where(
        #     (probs > 0.52) & low_vol_mask,  1,
        #     np.where((probs < 0.48) & low_vol_mask, -1, 0)
        # )
        signals = np.where(probs > 0.52, 1, np.where(probs < 0.48, -1, 0))
        strat_returns = signals * asset_returns
        excess_returns = strat_returns - (0.04 / 52) # assume 4% annual risk-free rate; 52 trading periods

        if excess_returns.std() == 0:
            sharpe = 0
        else:
            sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(52)

        # correct big moves
        correct_big_moves = np.mean(signals[(np.abs(asset_returns) > 0.02)] == 
                                    np.sign(asset_returns[(np.abs(asset_returns) > 0.02)]))

        all_results.append({'fold': fold_num+1, 'IC': round(ic,4), 'accuracy': round(acc,4), 'sharpe': round(sharpe, 4),\
                            'Accuracy on >2% moves': round(correct_big_moves, 3)})
        print(f"Fold {fold_num+1}: IC={ic:.4f}, Acc={acc:.4f}")
        print(f"Total test rows: {len(signals)}")
        print(f"Rows with signal != 0: {(signals != 0).sum()}")
        print(f"vol_5d threshold: {vol_5d_thresh:.4f}")
        print(f"probs distribution: min={probs.min():.3f}, max={probs.max():.3f}, mean={probs.mean():.3f}")
        X_te = X_te.fillna(0)

        baseline_ic = np.corrcoef(X_te['vol_5d'], y_te)[0,1]
        print(f"Baseline IC (vol only): {baseline_ic:.4f}")
        # Directional accuracy of baseline (vol_5d only)
        baseline_preds = (X_te['vol_5d'] > X_te['vol_5d'].median()).astype(int)
        baseline_acc = accuracy_score(y_te, baseline_preds)
        print(f"Baseline accuracy: {baseline_acc:.4f}")

        daily_ic = daily_ic_series(model, X_te, y_te, test_df)
        all_daily_ic.append(daily_ic)

    results_df = pd.DataFrame(all_results)
    full_daily_ic = pd.concat(all_daily_ic)

    print(f"\nMean daily IC: {full_daily_ic.mean():.4f}")
    print(f"Daily IC Std:  {full_daily_ic.std():.4f}")
    print(f"Daily ICIR:    {full_daily_ic.mean() / full_daily_ic.std():.4f}")
    print(f"N days:        {len(full_daily_ic)}")

        
    return pd.DataFrame(all_results), full_daily_ic, model, X_te, y_te

In [50]:
feat_cols = [
    'vol_5d', 'vol_20d', 'rsi_14', #'news_sentiment', 'pct_positive', 'news_count',
    'gpu_mentions', 'capex_up_score', #'capex_down_score',
    'capex_net', 'competitor_mentions', 'ai_sentence_ratio'
]

results_df, full_daily_ic, final_model, X_te, y_te = train_evaluate(splits, feat_cols)
results_df

[LightGBM] [Info] Number of positive: 1603, number of negative: 1892
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 799
[LightGBM] [Info] Number of data points in the train set: 3495, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1: IC=0.2181, Acc=0.5700
Total test rows: 4395
Rows with signal != 0: 3721
vol_5d threshold: 0.0286
probs distribution: min=0.228, max=0.727, mean=0.508
Baseline IC (vol only): 0.0436
Baseline accuracy: 0.5295
[LightGBM] [Info] Number of positive: 3346, number of negative: 3884
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000383 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[

,fold,IC,accuracy,sharpe,Accuracy on >2% moves
0,1,0.2181,0.5700,2.8148,0.634
1,2,0.2493,0.5956,2.4868,0.664
2,3,0.2268,0.5879,2.9829,0.693
3,4,0.2422,0.5919,2.4558,0.614
4,5,0.2695,0.6066,2.9413,0.653


In [26]:
results_df.to_csv('../../outputs/results_df.csv', index=False)
results_df.to_latex('../../outputs/results_table.tex', index=False, float_format='%.4f')

In [48]:
icir = results_df['IC'].mean() / results_df['IC'].std()
print(f"Median IC: {results_df['IC'].median():.4f}")
print(f'Mean IC: {results_df['IC'].mean():.4f}' )
print(f"IC Std:  {results_df['IC'].std():.4f}")
print(f"ICIR:    {icir:.4f}")

Median IC: 0.2422
Mean IC: 0.2412
IC Std:  0.0201
ICIR:    12.0272


In [ ]:
feat_cols = [
    'vol_5d', 'vol_20d', 'rsi_14', #'news_sentiment', 'pct_positive', 'news_count',
    'gpu_mentions', 'capex_up_score', #'capex_down_score',
    'capex_net', 'competitor_mentions', 'ai_sentence_ratio'
]

importance = pd.Series(
    final_model.feature_importances_, 
    index=feat_cols
).sort_values(ascending=False)

print(importance)

vol_20d                379
rsi_14                 337
vol_5d                 295
ai_sentence_ratio      203
gpu_mentions           109
capex_up_score          56
competitor_mentions     55
capex_net                6
news_sentiment           0
pct_positive             0
news_count               0
capex_down_score         0
dtype: int32


In [10]:
importance.to_csv('../../outputs/feature_importance.csv')

### Testing

Note: Some cells below are repeated from `data_cleaning.ipynb`.

In [11]:
tickers = pd.read_csv('../../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
all_tickers = [t for group in ai_tickers.values() for t in group]
all_tickers

['NVDA',
 'AMD',
 'INTC',
 'ASML',
 'TSM',
 'AVGO',
 'MRVL',
 'ARM',
 'SMCI',
 'MSFT',
 'GOOGL',
 'AMZN',
 'META',
 'PLTR',
 'AI',
 'SOUN',
 'DELL',
 'ANET',
 'HPE']

In [12]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [15]:
end = date.today() + timedelta(days=1)
start = end - timedelta(days=400)  # extra buffer for 252-day rolling windows

prices_live = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start=start,
    end=end,
    auto_adjust=True,
    progress=False
)

close_live = prices_live['Close']
volume_live = prices_live['Volume']

# Build features for each ticker
live_features = []
for ticker in all_tickers:
    if ticker not in close_live.columns:
        print(f"Missing: {ticker}")
        continue
    feats = make_features(
        close=close_live[ticker],
        volume=volume_live[ticker],
        soxx=close_live['SOXX'],
        qqq=close_live['QQQ']
    )
    feats['Ticker'] = ticker
    live_features.append(feats)

live_df = pd.concat(live_features).reset_index()
live_df = live_df.set_index(['Date', 'Ticker'])

# Get the most recent row per ticker (should be June 18)
latest = live_df.groupby('Ticker').tail(1)
print(f"Latest date: {latest.index.get_level_values('Date').unique()}")
print(f"Tickers: {latest.index.get_level_values('Ticker').tolist()}")

feat_cols = [
    'vol_5d', 'vol_20d', #'rsi_14', #'news_sentiment', 'pct_positive', 'news_count',
    'gpu_mentions', 'capex_up_score', #'capex_down_score',
    'capex_net', 'competitor_mentions', 'ai_sentence_ratio'
]

for col in feat_cols:
    if col not in latest.columns:
        latest[col] = 0
latest[feat_cols] = latest[feat_cols].fillna(0)

probs_today = final_model.predict_proba(latest[feat_cols])[:, 1]

ranking = pd.Series(probs_today, index=latest.index.get_level_values('Ticker'))
ranking = ranking.sort_values(ascending=False)
print("\nPredicted outperformers for next 5 trading days:")
print(ranking)

Latest date: DatetimeIndex(['2026-09-22'], dtype='datetime64[ns]', name='Date', freq=None)
Tickers: ['NVDA', 'AMD', 'INTC', 'ASML', 'TSM', 'AVGO', 'MRVL', 'ARM', 'SMCI', 'MSFT', 'GOOGL', 'AMZN', 'META', 'PLTR', 'AI', 'SOUN', 'DELL', 'ANET', 'HPE']

Predicted outperformers for next 5 trading days:
Ticker
INTC     0.524748
META     0.511698
SMCI     0.510014
SOUN     0.507917
AMD      0.507663
ARM      0.505927
AMZN     0.501814
ASML     0.497686
AVGO     0.497686
HPE      0.496608
DELL     0.495429
MSFT     0.491584
GOOGL    0.491584
TSM      0.491584
AI       0.488105
MRVL     0.480754
PLTR     0.478511
ANET     0.471740
NVDA     0.422001
dtype: float64


/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_88991/92889084.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  latest[col] = 0
/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_88991/92889084.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  latest[col] = 0
/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_88991/92889084.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

In [16]:
colors = ['#2ecc71' if p > 0.5 else '#e74c3c' for p in ranking.values[::-1]]

fig = go.Figure(go.Bar(
    x=ranking.values[::-1],
    y=ranking.index[::-1],
    orientation='h',
    marker_color=colors,
    text=[f'{p:.3f}' for p in ranking.values[::-1]],
    textposition='outside'
))

fig.add_vline(
    x=0.5,
    line_dash='dash',
    line_color='black',
    annotation_text='Decision threshold (0.5)',
    annotation_position='top'
)

fig.update_layout(
    title=f'AI Sector Stock Rankings — Next 5 Trading Days<br>{latest.index.get_level_values("Date").unique()[0].date()}',
    xaxis_title='Predicted Outperformance Probability',
    xaxis=dict(range=[0, 0.75]),
    height=600,
    width=900,
    plot_bgcolor='white',
    showlegend=False
)

fig.show()

### SHAP Plot

In [17]:
results_df, final_model, X_test, Y_te = train_evaluate(splits)
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

[LightGBM] [Info] Number of positive: 1603, number of negative: 1892
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002268 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 544
[LightGBM] [Info] Number of data points in the train set: 3495, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Fold 1: IC=0.0295, Acc=0.5119
Total test rows: 4395
Rows with signal != 0: 0
vol_5d threshold: 0.0286
probs distribution: min=0.480, max=0.517, mean=0.500
Baseline IC (vol only): 0.0436
Baseline accuracy: 0.5295
[LightGBM] [Info] Number of positive: 3346, number of negative: 3884
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[Lig

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/shap/explainers/_tree.py:583: UserWarning:

LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray



In [18]:
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.gcf().set_size_inches(4,6)
plt.savefig('../../outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()

In [19]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[1000],
        base_values=explainer.expected_value,
        data=X_test.round(3).iloc[1000],
        feature_names=feat_cols
    ),
    max_display=12,
    show=False
)
plt.savefig('../../outputs/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.close()

### Other Visualizations

In [20]:
# IC by fold bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width/2, results_df['IC'], width, label='Model IC', color='#2ecc71')
ax.bar(x + width/2, [0.0436, 0.0316, 0.0717, 0.0673, 0.0566], width, label='Baseline IC', color='#95a5a6')
ax.set_xlabel('Fold')
ax.set_ylabel('Information Coefficient')
ax.set_title('Model IC vs Volatility-Only Baseline by Fold')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/ic_by_fold.png', dpi=150, bbox_inches='tight')

In [21]:
# Cumulative IC Over Time
fig, ax = plt.subplots(figsize=(10, 5))
folds = [f'Fold {i+1}' for i in range(5)]
model_ics = results_df['IC'].values
baseline_ics = [0.0436, 0.0316, 0.0717, 0.0673, 0.0566]

ax.plot(folds, model_ics, marker='o', color='#2ecc71', linewidth=2, label='Model IC')
ax.plot(folds, baseline_ics, marker='s', color='#95a5a6', linewidth=2, linestyle='--', label='Baseline IC')
ax.fill_between(folds, model_ics, baseline_ics, alpha=0.2, color='#2ecc71')
ax.set_xlabel('Fold')
ax.set_ylabel('Information Coefficient')
ax.set_title('Model vs Baseline IC Across Walk-Forward Folds')
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/ic_over_folds.png', dpi=150, bbox_inches='tight')

### Streamlit Dashboard


In [22]:
st.set_page_config(page_title="AI Stock Forecasting", layout="wide")
st.title("AI Sector Stock Forecasting Dashboard")
st.caption("Hybrid model: price features + FinBERT news sentiment + earnings keyword signals")

# Top metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("Walk-forward IC",    "",  "+0.038 vs price-only baseline")
col2.metric("Directional accuracy","54.8%", "+2.1% vs baseline")
col3.metric("Sharpe (L/S sim)",   "1.31")
col4.metric("Universe",           "19 AI stocks")

# Sector heatmap
st.subheader("AI sector performance — last 30 days")
tickers = ['NVDA','AMD','MSFT','GOOGL','META','AMZN','PLTR','AVGO','TSM']
data = yf.download(tickers, period='30d', auto_adjust=True)['Close']
returns_30d = data.pct_change(len(data)-1).iloc[-1].sort_values(ascending=False)
fig = go.Figure(go.Bar(x=returns_30d.index, y=returns_30d.values*100,
    marker_color=['#185FA5' if v>0 else '#993C1D' for v in returns_30d.values]))
fig.update_layout(yaxis_title='30-day return (%)', height=300)
st.plotly_chart(fig, use_container_width=True)

# SHAP chart
st.subheader("What drives predictions — SHAP feature importance")
st.image('../../outputs/shap_summary.png')

2026-09-22 11:40:35.349 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.365 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-22 11:40:35.585 
  command:

    streamlit run /Users/heiditam/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-09-22 11:40:35.585 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-22 11:40:35.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ign

DeltaGenerator()